In [2]:
from datasets import load_dataset

ds = load_dataset("framolfese/2WikiMultihopQA", trust_remote_code=True)

print(ds["train"].features)
print(ds["train"][0])

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/165M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/29.5M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/28.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/167454 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/12576 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12576 [00:00<?, ? examples/s]

{'id': Value(dtype='string', id=None), 'question': Value(dtype='string', id=None), 'answer': Value(dtype='string', id=None), 'type': Value(dtype='string', id=None), 'evidences': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None), 'supporting_facts': {'title': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'sent_id': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None)}, 'context': {'title': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'sentences': Sequence(feature=Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), length=-1, id=None)}}
{'id': '13f5ad2c088c11ebbd6fac1f6bf848b6', 'question': 'Are director of film Move (1970 Film) and director of film Méditerranée (1963 Film) from the same country?', 'answer': 'no', 'type': 'bridge_comparison', 'evidences': [['Move (1970 film)', 'director', 'Stuart Rosenberg'], ['Méditerranée (1963 film)', 'director', '

In [3]:
label2id = {"comparison": 0, "inference": 1, "compositional": 2, "bridge_comparison": 3}
id2label = {v: k for k, v in label2id.items()}

def preprocess(example):
    example["label"] = label2id[example["type"]]
    return example

ds = ds.map(preprocess)

Map:   0%|          | 0/167454 [00:00<?, ? examples/s]

Map:   0%|          | 0/12576 [00:00<?, ? examples/s]

Map:   0%|          | 0/12576 [00:00<?, ? examples/s]

In [12]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["question"], truncation=True, padding="max_length", max_length=128)

tokenized = ds.map(tokenize, batched=True)
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/167454 [00:00<?, ? examples/s]

Map:   0%|          | 0/12576 [00:00<?, ? examples/s]

Map:   0%|          | 0/12576 [00:00<?, ? examples/s]

In [15]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

args = TrainingArguments(
    output_dir="./2wiki-type-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.000000,0.000732,0.999920,0.999895
2,0.000000,0.000099,0.999920,0.999895
3,0.000000,0.000001,1.000000,1.000000


TrainOutput(global_step=15699, training_loss=0.0026173893392164305, metrics={'train_runtime': 2174.8222, 'train_samples_per_second': 230.99, 'train_steps_per_second': 7.219, 'total_flos': 1.6637240212862976e+16, 'train_loss': 0.0026173893392164305, 'epoch': 3.0})

In [23]:
results = trainer.evaluate(tokenized["test"])
print(results)

{'eval_loss': 0.0021868280600756407, 'eval_accuracy': 0.9998409669211196, 'eval_f1_macro': 0.9997911041269263, 'eval_runtime': 13.7524, 'eval_samples_per_second': 914.458, 'eval_steps_per_second': 14.325, 'epoch': 3.0}


In [30]:
from transformers import pipeline

clf = pipeline("text-classification", model="./2wiki-type-classifier-final")
clf("Which film came out earlier, Onnum Mindatha Bharya or Gold Of The Amazon Women?")

Device set to use cuda:0


[{'label': 'comparison', 'score': 0.9999997615814209}]